# Mixed trace-gallery experiment

This notebook reproduces the task-held-out mixed-gallery study in `reports/09-关联分析文献与混合候选实验.md`. It compares behavior-only rankings with an optional execution-mode score. A CTF task ID is only a same-challenge proxy label: it is not a shared attack activity, campaign, or actor label. Human terminal capture and Agent tool-call capture are confounded with the class.

The notebook uses public records already downloaded by the repository scripts. It does not execute commands from the traces.

## Fixed protocol

- 177 non-empty records in 40 benchmark-qualified task groups.
- Three task-grouped folds with primary seed `20261030`; five alternate seeds test split sensitivity.
- Each query is ranked against all other traces in its benchmark and held-out fold. Candidates are not filtered by execution mode.
- Pair training uses only task identity; mode/source fields are excluded from inference and pair construction.
- Primary metric: task-macro Hit@1; supporting metrics: MRR, Recall@5, nDCG@5. Confidence intervals use 5,000 benchmark-stratified task-cluster bootstrap draws, conditional on fixed folds.

Run the cells in order from the repository root, or let the first code cell locate the root from the notebook folder.

In [ ]:
from pathlib import Path
import json, subprocess, sys

ROOT = Path.cwd()
if (ROOT / 'research' / 'scripts').is_dir(): ROOT = ROOT / 'research'
if not (ROOT / 'scripts' / 'run_mixed_gallery.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'scripts' / 'run_mixed_gallery.py').exists(), f'Cannot find repository root from {Path.cwd()}'
print('Repository:', ROOT)
print('Python:', sys.version.split()[0])

## Recompute all reported splits

The run refreshes the JSON, per-task and per-query CSVs, out-of-fold mode predictions, and publication figures. Default analysis dependencies are pinned in `requirements.lock.txt`.

In [ ]:
completed = subprocess.run(
    [str(ROOT.parent / '.venv' / 'bin' / 'python'), 'scripts/run_mixed_gallery.py', '--bootstraps', '5000'],
    cwd=ROOT, check=True, text=True, capture_output=True
)
print(completed.stdout[-5000:])

## Main results

Point estimates are averaged within task, then tasks within benchmark, with equal weight for each benchmark. The five alternate splits are sensitivity checks over the same underlying records, not independent external replications.

In [ ]:
import pandas as pd
from IPython.display import display

result = json.loads((ROOT / 'results' / 'mixed_gallery.json').read_text())
primary = result['primary']
summary = primary['summary']
rows = []
for method, stats in summary.items():
    ci = stats['task_cluster_bootstrap_95_ci']['hit_at_1']
    rows.append({
        'method': method, 'Hit@1': stats['hit_at_1'],
        'Hit@1 95% CI': f'[{ci[0]:.3f}, {ci[1]:.3f}]',
        'MRR': stats['mrr'], 'Recall@5': stats['recall_at_5'],
        'nDCG@5': stats['ndcg_at_5'], 'tasks': stats['task_count'],
        'mean candidates': stats['candidate_count_mean'],
    })
pd.DataFrame(rows).sort_values('Hit@1', ascending=False).round(3)

In [ ]:
def hit_delta(name, source):
    item = source['pairwise'][name]['hit_at_1']
    lo, hi = item['task_cluster_bootstrap_95_ci']
    return {'comparison': f'mode_aware - {name}', 'delta': item['mode_aware_minus_baseline'],
            '95% CI': f'[{lo:.3f}, {hi:.3f}]', 'paired tasks': item['paired_tasks']}

primary_delta_rows = [hit_delta(name, primary['deltas']) for name in primary['summary'] if name != 'mode_aware']
robust = result['robustness_mean_across_splits']
robust_delta_rows = [hit_delta(name, robust['deltas']) for name in robust['summary'] if name != 'mode_aware']
print('Primary split')
display(pd.DataFrame(primary_delta_rows).round(3))
print('Mean of five alternate splits, averaged within each task first')
display(pd.DataFrame(robust_delta_rows).round(3))

In [ ]:
split_rows = []
for split in result['robustness']:
    delta = split['deltas']['hit_at_1']
    lo, hi = delta['task_cluster_bootstrap_95_ci']
    split_rows.append({'seed': split['seed'], 'delta Hit@1': delta['mode_aware_minus_supervised_similarity'],
                       '95% CI': f'[{lo:.3f}, {hi:.3f}]'})
pd.DataFrame(split_rows).round(3)

## Held-out mode identification diagnostics

The following confusion matrix and calibration plot describe prediction of the release's Human/Agent record labels. Because collection interface is aligned with those labels, they are not an estimate of general operator identity recognition.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'results' / 'figures' / 'mixed_gallery_retrieval.png')))
display(Image(filename=str(ROOT / 'results' / 'figures' / 'mixed_gallery_confusion.png')))
display(Image(filename=str(ROOT / 'results' / 'figures' / 'mixed_gallery_calibration.png')))

## Interpretation

On the primary split, adding mode features improves Hit@1 over the task-supervised behavior ranker by 3.6 percentage points, with the fixed-fold task bootstrap interval above zero. The estimate does not beat command cosine with a clear paired difference, and the five alternate split estimates average to −0.7 points with an interval crossing zero. Therefore the current result does **not** establish a stable association benefit.

The experiment tests same-task retrieval, not shared attacker or campaign linkage. A decisive follow-up needs repeated human, script, and Agent execution under the same sensor pipeline, independently assigned activity IDs, and an activity-level gallery.

In [ ]:
assert result['dataset']['records'] == 177
assert result['dataset']['tasks'] == 40
assert len(result['robustness']) == 5
assert all(f['train_tasks'] and f['test_tasks'] and not set(f['train_tasks']) & set(f['test_tasks']) for f in primary['folds'])
print('Audit passed: data counts, alternate splits, and task disjointness match the recorded protocol.')